# 06 – Neural-network (MLP) surrogate baseline

Trains a 2-layer MLP surrogate on the **same** rule-based dataset as SINDy,
then evaluates it in:
1. Open-loop one-step and rollout prediction (vs SINDy)
2. Closed-loop control via shooting MPC (scipy L-BFGS-B + PyTorch autograd)
3. Inference latency comparison

This notebook produces the `nn_mpc` row that joins the `closed_loop_benchmark.csv` table.

In [ ]:
from pathlib import Path
import sys

HERE = Path.cwd().resolve()
if (HERE / 'article_experiment_utils.py').exists():
    ARTICLE_DIR = HERE
    REPO_DIR = HERE.parent
else:
    ARTICLE_DIR = HERE / 'own-article'
    REPO_DIR = HERE
sys.path.insert(0, str(ARTICLE_DIR))
sys.path.insert(0, str(REPO_DIR))

from article_experiment_utils import *
OUT = results_dir()

In [ ]:
FAST_MODE = False  # set True for a quick smoke-test (1-day rollout, 100 epochs)

cfg = ExperimentConfig(n_days=60, start_date='2010-02-28', horizon=20)
rollout_days   = 1 if FAST_MODE else 14
nn_epochs      = 100 if FAST_MODE else 500
benchmark_date = '2010-04-15'

# ── Load training data (same as notebooks 01-03) ─────────────────────────────
train_path = OUT / 'datasets' / 'rbc_train_60d.npz'
train_data = load_dataset(train_path) if train_path.exists() \
    else collect_rule_based_dataset(cfg)
save_dataset(train_data, train_path)

train_subset = train_data.subset_steps(60 * cfg.steps_per_day)
print(f'Training rows: {len(train_subset.states)}')

# ── Load test data (same 30-day window as notebooks 01-03) ───────────────────
test_path = OUT / 'datasets' / 'rbc_test_30d_2010_04_15.npz'
cfg_test = ExperimentConfig(n_days=30, start_date=benchmark_date)
test_data = load_dataset(test_path) if test_path.exists() \
    else collect_rule_based_dataset(cfg_test, n_days=30, start_date=benchmark_date)
save_dataset(test_data, test_path)
print(f'Test rows: {len(test_data.states)}')

## 1. Train MLP surrogate

In [ ]:
import time

t0 = time.perf_counter()
nn_bundle = fit_nn_surrogate(
    train_subset,
    feature_variant='physics',
    hidden_sizes=[64, 64],
    epochs=nn_epochs,
    lr=1e-3,
    batch_size=512,
    period=float(cfg.period),
    metadata={'label': 'nn_mpc'},
)
train_time = time.perf_counter() - t0

save_bundle(nn_bundle, OUT / 'models' / 'nn_mpc_physics.pkl')
print(f'Train time: {train_time:.1f}s  |  Final MSE (scaled): {nn_bundle.train_loss:.5f}')
print(f'Params: {sum(w.size for w in nn_bundle.weights) + sum(b.size for b in nn_bundle.biases)}')

## 2. One-step and rollout prediction accuracy vs SINDy

In [ ]:
import pandas as pd

# Load SINDy bundles trained in notebook 01/02
sindy_raw_path    = OUT / 'models' / 'raw_sindy_mpc.pkl'
sindy_phys_path   = OUT / 'models' / 'physics_sindy_mpc.pkl'

prediction_rows = []

# MLP
nn_metrics = evaluate_nn_surrogate(nn_bundle, test_data, rollout_horizons=(1, 4, 20, 96))
nn_metrics.insert(0, 'method', 'nn_mpc_physics')
prediction_rows.append(nn_metrics)

# SINDy baselines (if available)
for path, label in [(sindy_raw_path, 'raw_sindy'), (sindy_phys_path, 'physics_sindy')]:
    if path.exists():
        b = load_bundle(path)
        m = evaluate_sindy(b, test_data, rollout_horizons=(1, 4, 20, 96))
        m.insert(0, 'method', label)
        prediction_rows.append(m)

pred_df = pd.concat(prediction_rows, ignore_index=True)
save_table(pred_df, OUT / 'tables' / 'nn_vs_sindy_prediction.csv')

# Pivot: one-step RMSE comparison
pivot = pred_df[pred_df['metric_scope'] == 'one_step'].pivot(
    index='method', columns='state', values='rmse'
).round(3)
print('One-step RMSE on test set:')
pivot

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
state_labels = {'t_in': 'Temperature RMSE (°C)', 'co2': 'CO₂ RMSE (ppm)', 'rh': 'RH RMSE (%)'}
for ax, state in zip(axes, ['t_in', 'co2', 'rh']):
    for method, grp in pred_df[(pred_df['metric_scope'] == 'rollout') & (pred_df['state'] == state)].groupby('method'):
        ax.plot(grp['horizon'], grp['rmse'], marker='o', label=method)
    ax.set_xlabel('Rollout horizon (steps)')
    ax.set_ylabel(state_labels[state])
    ax.set_title(state_labels[state])
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8)

fig.suptitle('Open-loop rollout RMSE: MLP vs SINDy', fontsize=12)
fig.tight_layout()
save_figure(fig, OUT / 'figures' / 'nn_vs_sindy_rollout.png')
plt.show()

## 3. Closed-loop control: MLP-MPC vs SINDy-MPC vs Rule-based

> **Note:** MLP-MPC uses scipy L-BFGS-B with PyTorch autograd gradients.
> Expect ~20–40 min for 14 days on CPU. Set `FAST_MODE = True` for a 1-day test.

In [ ]:
rollouts_all = {}

# Rule-based (load if pre-computed)
rbc_path = OUT / 'tables' / 'closed_loop_rollout_rule_based_paper.csv'
rollouts_all['rule_based'] = (
    pd.read_csv(rbc_path) if rbc_path.exists()
    else rollout_rule_based(cfg, n_days=rollout_days, start_date=benchmark_date, noise_scale=0.0)
)

# Physics SINDy-MPC (load if pre-computed)
pi_path = OUT / 'tables' / 'closed_loop_rollout_physics_sindy_mpc_paper.csv'
rollouts_all['physics_sindy_mpc'] = (
    pd.read_csv(pi_path) if pi_path.exists()
    else rollout_mpc(load_bundle(sindy_phys_path), cfg, n_days=rollout_days,
                     start_date=benchmark_date, objective='full')
)

print('Starting MLP-MPC rollout (this takes a while on CPU)...')
t0 = time.perf_counter()
df_nn = rollout_mpc_nn(
    nn_bundle, cfg, n_days=rollout_days, start_date=benchmark_date, horizon=cfg.horizon
)
print(f'MLP-MPC rollout done in {(time.perf_counter()-t0)/60:.1f} min  |  steps: {len(df_nn)}')
rollouts_all['nn_mpc'] = df_nn
save_table(df_nn, OUT / 'tables' / 'closed_loop_rollout_nn_mpc_paper.csv')

In [ ]:
bench = benchmark_rollouts(rollouts_all)

# Merge with existing benchmark CSV to get a unified comparison table
existing_bench_path = OUT / 'tables' / 'closed_loop_benchmark.csv'
if existing_bench_path.exists():
    existing = pd.read_csv(existing_bench_path)
    # Remove rule_based and physics_sindy_mpc rows to avoid duplication, keep raw_sindy_mpc
    extra = existing[existing['method'] == 'raw_sindy_mpc']
    bench = pd.concat([bench, extra], ignore_index=True)

save_table(bench, OUT / 'tables' / 'closed_loop_benchmark_with_nn.csv')

cols = ['method', 'temp_rmse', 'co2_rmse', 'comfort_pct', 'energy_proxy_sum',
        'temp_low_violations', 'rh_excess_area']
bench[[c for c in cols if c in bench.columns]].round(2)

In [ ]:
plot_rollout_comparison(rollouts_all, OUT / 'figures', suffix='_with_nn')

## 4. Inference latency: SINDy vs MLP (one-step prediction)

In [ ]:
sindy_bundles = {}
if sindy_raw_path.exists():
    sindy_bundles['raw_sindy'] = load_bundle(sindy_raw_path)
if sindy_phys_path.exists():
    sindy_bundles['physics_sindy'] = load_bundle(sindy_phys_path)

nn_bundles = {'nn_mpc_physics': nn_bundle}

timing_df = measure_inference_time(
    sindy_bundles, nn_bundles, test_data, n_samples=1000
)
save_table(timing_df, OUT / 'tables' / 'inference_timing.csv')
timing_df.round(4)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 3))
colors = ['#2196F3'] * len(sindy_bundles) + ['#FF9800'] * len(nn_bundles)
ax.barh(timing_df['method'], timing_df['mean_ms'], xerr=timing_df['std_ms'],
        color=colors, capsize=4, edgecolor='black', linewidth=0.6)
ax.set_xlabel('One-step inference time (ms)')
ax.set_title('Inference latency: SINDy (blue) vs MLP (orange)')
ax.grid(True, axis='x', alpha=0.3)
fig.tight_layout()
save_figure(fig, OUT / 'figures' / 'inference_timing.png')
plt.show()
print('\nSpeedup SINDy vs MLP:')
for _, row in timing_df.iterrows():
    print(f"  {row['method']}: {row['mean_ms']:.4f} ms (median {row['median_ms']:.4f} ms)")

## 5. Model complexity comparison

Key paper claim: SINDy is orders of magnitude more compact than MLP.

In [ ]:
import numpy as np

nn_params = sum(w.size for w in nn_bundle.weights) + sum(b.size for b in nn_bundle.biases)

model_summary = []
for label, b in sindy_bundles.items():
    nz = int(np.count_nonzero(b.model.coefficients()))
    total = int(b.model.coefficients().size)
    model_summary.append({
        'method': label,
        'nonzero_params': nz,
        'total_params': total,
        'sparsity': round(1 - nz/total, 3),
    })

model_summary.append({
    'method': 'nn_mpc_physics',
    'nonzero_params': nn_params,
    'total_params': nn_params,
    'sparsity': 0.0,
})

complexity_df = pd.DataFrame(model_summary)
save_table(complexity_df, OUT / 'tables' / 'model_complexity.csv')
complexity_df